# CYP51 true molecular-cartoon MD movies — CAL-028 and CAL-015

**This replaces the earlier Matplotlib Cα trace with genuine PyMOL secondary-structure cartoons:** protein α-helices and β-sheets as ribbons, ligand and pocket residues as sticks, heme/iron and time-resolved polar-contact lines. Both movies use your actual 50 ns simulation frames (not interpolated structures).

**Runtime:** Google Colab CPU. **Input:** files already downloaded into `MyDrive/CYP51_MD_animation_sources/`. **Output:** `MyDrive/CYP51_MD_3D_cartoon_videos/CAL028/` and `CAL015/`.

The old `*_aligned_pocket_50ns.dcd` file retained Cα atoms only for much of the protein, so it cannot produce a correct helix/sheet cartoon. This notebook extracts the **full N, CA, C, O backbone** from your existing 50 original DCDs, aligns it, and renders PyMOL frames. It caches each segment separately so an interrupted extraction is resumable.

**Outputs per compound:** `.mp4` true-cartoon movie, `.png` first-frame preview, `.pse` PyMOL interactive session, full-backbone aligned `.pdb` + `.dcd`. To check the style quickly, set `MAX_FRAMES=25` in Cell 4, render, then set `MAX_FRAMES=200` for the complete movie. Keep `RAY_TRACE=0` for fast full-color 3D cartoon; `1` for ray-traced production rendering. The videos are 960 × 720, H.264, 15 fps.

In [ ]:
# Cell 1 — Install PyMOL, MDTraj and a headless display for genuine helix/sheet cartoon rendering.
# PyMOL is a real molecular-graphics renderer, not a Matplotlib line/scatter plot.
!apt-get -qq update > /dev/null
!DEBIAN_FRONTEND=noninteractive apt-get -qq install -y xvfb libglu1-mesa libgl1 ffmpeg > /dev/null
!pip -q install 'pymol-open-source==3.2.0a0.post5859' 'mdtraj>=1.11' pyvirtualdisplay pillow
import pymol2, mdtraj as md
print('PyMOL and MDTraj are available.')


In [ ]:
# Cell 2 — Mount Drive. We will never overwrite the original 50 DCD files.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
root=Path('/content/drive/MyDrive/CYP51_MD_animation_sources')
for sys in ('CAL028','CAL015'):
    assert (root/sys/'system.prmtop').exists(),f'{sys}: topology missing'
    assert len(list((root/sys).glob('ns_*.dcd')))==50,f'{sys}: 50 DCDs not found'
    print(f'{sys}: 50 segments + topology available')


In [ ]:
# Cell 3 — Read original AMBER trajectories and create a FULL-BACKBONE aligned cartoon source.
# Unlike the earlier C-alpha-only pocket file, this keeps N, CA, C, and O for every protein residue.
from pathlib import Path
import shutil
import tempfile
import numpy as np
import mdtraj as md

ROOT = Path('/content/drive/MyDrive')
SOURCE = ROOT/'CYP51_MD_animation_sources'
OUT = ROOT/'CYP51_MD_3D_cartoon_videos'
SYSTEMS = ('CAL028', 'CAL015')
STRIDE = 25                 # 4 real frames/ns x 50 ns = 200 animation frames
PDB_START = 45              # CYP51 5TZ1 residue numbering
POCKET = {
    'CAL028': [118,132,233,307,308,376,377,378,379,380,507,508],
    'CAL015': [118,121,126,131,132,228,233,376,377,508],
}
INTERACTIONS = {
    'CAL028': [('TYR118','OH','O4'),('ILE379','N','O3'),
               ('PHE380','N','O4'),('SER378','N','O3')],
    'CAL015': [('MET508','O','N4')],
}
OUT.mkdir(parents=True,exist_ok=True)

def heavy(a):
    return a.element is None or a.element.symbol.upper() not in ('H', 'D')

def describe_topology(top, system):
    protein = [r for r in top.residues
               if {'N','CA','C','O'}.issubset({a.name for a in r.atoms})
               and r.name.upper() not in ('HEM','LIG')]
    assert len(protein)==484, f'{system}: expected 484 protein residues, found {len(protein)}'
    residues = {PDB_START+i:r for i,r in enumerate(protein)}
    ca=np.asarray([next(a.index for a in r.atoms if a.name=='CA') for r in protein],dtype=int)
    backbone=[a.index for r in protein for a in r.atoms if a.name in ('N','CA','C','O')]
    sidechain=[a.index for n in POCKET[system] for a in residues[n].atoms if heavy(a)]
    lig=[a.index for a in top.atoms if a.residue.name=='LIG' and heavy(a)]
    hem=[a.index for a in top.atoms if a.residue.name=='HEM' and heavy(a)]
    assert len(lig)==32, f'{system}: expected 32 ligand heavy atoms, found {len(lig)}'
    assert len(hem)>=35, f'{system}: expected HEM atoms, found {len(hem)}'
    ids=np.unique(np.r_[backbone,sidechain,lig,hem]).astype(int)
    print(f'{system}: {len(ids)} selected atoms, including all 484 protein backbones, '
          f'{len(lig)} ligand heavy atoms, {len(hem)} heme heavy atoms')
    return ids,ca

def align_selected(mobile_ca, ref_ca, selected_xyz):
    a=mobile_ca.mean(0); b=ref_ca.mean(0)
    u,s,vh=np.linalg.svd((mobile_ca-a).T@(ref_ca-b))
    diagonal=np.diag([1.,1.,np.sign(np.linalg.det(u@vh))])
    rot=u@diagonal@vh
    return (selected_xyz-a)@rot+b

def make_cartoon_source(system):
    src=SOURCE/system; dest=OUT/system
    dest.mkdir(parents=True, exist_ok=True)
    topology_path=src/'system.prmtop'
    if not topology_path.exists():
        raise FileNotFoundError(f'Missing topology: {topology_path}')
    pdb=dest/f'{system}_full_backbone_cartoon.pdb'
    dcd=dest/f'{system}_full_backbone_cartoon_50ns.dcd'
    if pdb.is_file() and dcd.is_file():
        t=md.load_dcd(str(dcd),top=str(pdb))
        prot_count=len([r for r in t.top.residues if r.name not in ('HEM','LIG')])
        if t.n_frames==200 and prot_count==484 and t.top.select('name N').size>=484:
            print(f'{system}: REUSING saved full-backbone cartoon trajectory ({t.n_frames} frames)')
            return pdb,dcd
        print(f'{system}: cached source does not pass validation; rebuilding.')
    top=md.load_prmtop(str(topology_path))
    ids,ca=describe_topology(top,system)
    ca_positions={int(a):i for i,a in enumerate(ids)}
    ca_local=np.array([ca_positions[int(a)] for a in ca],dtype=int)
    cache=dest/f'_backbone_cache_stride_{STRIDE}'
    cache.mkdir(exist_ok=True)
    ref=None; all_xyz=[]; all_times=[]
    for ns in range(1,51):
        path=cache/f'ns_{ns:03d}.npz'
        if path.exists():
            with np.load(path) as saved:
                seg=saved['xyz']; times=saved['times']
            if seg.ndim==3 and seg.shape[1]==len(ids) and seg.shape[0]==4:
                # Stored frames are already aligned against ns_001 frame 0.
                all_xyz.append(seg);all_times.append(times)
                if ns%10==0: print(f'  {system}: cached {ns}/50 ns')
                continue
            path.unlink()
        dcdf=src/f'ns_{ns:03d}.dcd'
        if not dcdf.exists(): raise FileNotFoundError(dcdf)
        with tempfile.TemporaryDirectory() as td:
            local=Path(td)/'seg.dcd';shutil.copyfile(dcdf,local)
            traj=md.load_dcd(str(local),top=top,stride=STRIDE)
        if traj.n_frames!=4:
            raise RuntimeError(f'{system} ns {ns}: expected 4 frames; got {traj.n_frames}')
        if ref is None:
            # Reference can be recovered from ns_001 even if ns_001 is cached.
            with tempfile.TemporaryDirectory() as td:
                local=Path(td)/'first.dcd'
                shutil.copyfile(src/'ns_001.dcd',local)
                first=md.load_dcd(str(local),top=top,stride=STRIDE)
            ref=first.xyz[0,ca,:].copy()
        aligned=np.stack([align_selected(traj.xyz[j,ca,:],ref,traj.xyz[j,ids,:])
                           for j in range(traj.n_frames)],axis=0).astype(np.float32)
        ts=(ns-1)+(np.arange(traj.n_frames)*STRIDE+1)/100.0
        np.savez_compressed(path,xyz=aligned,times=ts)
        all_xyz.append(aligned); all_times.append(ts)
        if ns%10==0: print(f'  {system}: processed {ns}/50 ns')
    xyz=np.concatenate(all_xyz,axis=0)
    times=np.concatenate(all_times)
    sub=top.subset(ids.tolist())
    # Restore the 5TZ1 residue numbering and normalize AMBER histidine names.
    for r in sub.residues:
        if r.name=='HEM':r.resSeq=900
        elif r.name=='LIG':r.resSeq=901
        else:
            r.resSeq=r.index+PDB_START
            if r.name in ('HIE','HID','HIP'):r.name='HIS'
    result=md.Trajectory(xyz,topology=sub,time=times*1000)
    result[0].save_pdb(str(pdb))
    result.save_dcd(str(dcd))
    test=md.load_dcd(str(dcd),top=str(pdb))
    assert test.n_frames==200 and test.n_atoms==len(ids)
    print(f'{system}: VERIFIED {test.n_frames} frames; {test.n_atoms} atoms; file {dcd.name}')
    return pdb,dcd

source_files={}
for system in SYSTEMS:
    print('\n=== Preparing genuine 3D CARTOON SOURCE for',system,'===')
    source_files[system]=make_cartoon_source(system)
print('\nFull-backbone animated sources ready (original Drive trajectories unchanged).')


In [ ]:
# Cell 4 — PyMOL's genuine molecular CARTOON rendering (helices, sheets and loops).
# This is NOT Matplotlib's 3D scatter/CA spaghetti representation.
import subprocess
from pathlib import Path
from PIL import Image,ImageDraw,ImageFont
from pyvirtualdisplay import Display
import pymol2

FPS = 15
VIDEO_WIDTH, VIDEO_HEIGHT = 960, 720  # both even, avoids FFmpeg error
# Set to 1 for ray-traced quality, or 0 to render considerably faster in Xvfb.
RAY_TRACE = 0
# Set to 25 for a quick style check, then 200 for the full 50 ns animation.
MAX_FRAMES = 200

font_path='/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf'
def add_caption(path, system, ns, frame_index, nframes):
    image=Image.open(path).convert('RGB')
    draw=ImageDraw.Draw(image,'RGBA')
    draw.rounded_rectangle((18,15,345,79),radius=12,fill=(255,255,255,205))
    try:
        f1=ImageFont.truetype(font_path,21)
        f2=ImageFont.truetype(font_path,16)
    except OSError:
        f1=ImageFont.load_default(); f2=f1
    draw.text((32,21),f'{system} | CYP51 binding pocket',font=f1,fill=(20,28,40,255))
    draw.text((33,49),f'Production time: {ns:.2f} / 50.00 ns',font=f2,fill=(50,63,75,255))
    image.save(path)

def produce_cartoon(system, pdb, dcd):
    folder=OUT/system
    video=folder/f'{system}_CYP51_50ns_TRUE_cartoon.mp4'
    preview=folder/f'{system}_cartoon_preview.png'
    session=folder/f'{system}_cartoon_interactive.pse'
    # Use a virtual X server: real PyMOL rendering on the Colab CPU, no GPU required.
    display=Display(visible=0,size=(VIDEO_WIDTH,VIDEO_HEIGHT),color_depth=24)
    display.start()
    p=pymol2.PyMOL()
    p.start()
    c=p.cmd
    frames_dir=Path('/content')/f'{system}_pymol_frames'
    frames_dir.mkdir(parents=True,exist_ok=True)
    try:
        c.reinitialize()
        c.set('defer_builds_mode',3)
        c.load(str(pdb),'complex')
        try:
            c.load_traj(str(dcd),'complex',state=1)
            total=c.count_states('complex')
            if total != 200:
                raise RuntimeError(f'PyMOL DCD reader returned {total} states instead of 200')
        except Exception as e:
            # A few PyMOL wheel builds omit the Molfile DCD plugin.
            # Fallback: load real MD frames as individual PDB states, not interpolations.
            print(f'{system}: PyMOL DCD plugin unavailable ({e}); using MDTraj frame fallback')
            c.delete('complex')
            from pathlib import Path as _Path
            t=md.load_dcd(str(dcd),top=str(pdb))
            with tempfile.TemporaryDirectory() as tmp:
                for j in range(t.n_frames):
                    framefile=_Path(tmp)/'frame.pdb'
                    t[j].save_pdb(str(framefile))
                    c.load(str(framefile),'complex',state=j+1)
                    if (j+1)%50==0:
                        print(f'  {system}: loaded {j+1}/{t.n_frames} PyMOL states')
            total=c.count_states('complex')
            if total != 200:
                raise RuntimeError(f'Failed to load 200 PyMOL states; found {total}')
        c.bg_color('white')
        c.set('orthoscopic',1)
        c.set('depth_cue',0)
        c.set('antialias',2)
        c.set('specular',0.15)
        c.set('ray_shadows',0)
        c.set('ray_opaque_background',1)
        c.set('cartoon_fancy_helices',1)
        c.set('cartoon_flat_sheets',1)
        c.set('cartoon_smooth_loops',1)
        c.set('cartoon_transparency',0.13)
        c.set('stick_radius',0.17)
        c.set('sphere_scale',0.46)
        c.set('label_color','black')
        c.set('label_size',15)
        c.set('dash_color','yellow')
        c.set('dash_width',3)
        c.set('dash_gap',0.32)
        c.set('dash_length',0.2)
        c.hide('everything','all')
        c.select('protein','complex and polymer.protein')
        if c.count_atoms('protein')<1800:
            raise RuntimeError('Backbone protein selection failed. Verify the full-backbone source.')
        c.dss('protein')
        c.show('cartoon','protein')
        c.color('gray80','protein')
        c.select('ligand','complex and resn LIG')
        c.select('heme','complex and resn HEM')
        c.select('fe','heme and name FE')
        c.select('pocket','complex and resi '+'+'.join(map(str,POCKET[system]))+' and polymer.protein')
        c.show('sticks','pocket and not name N+CA+C+O')
        c.color('gray55','pocket and elem C')
        c.show('sticks','ligand')
        c.set_color('ligand_cyan',[0.07,0.69,0.81])
        c.set_color('ligand_magenta',[0.80,0.24,0.62])
        c.color('ligand_cyan' if system=='CAL028' else 'ligand_magenta','ligand and elem C')
        c.color('red','ligand and elem O')
        c.color('blue','ligand and elem N')
        c.color('yellow','ligand and elem S')
        c.show('sticks','heme and not name FE')
        c.color('salmon','heme and elem C')
        c.color('red','heme and elem O')
        c.color('blue','heme and elem N')
        c.show('spheres','fe')
        c.color('orange','fe')
        # Labels follow the restored PDB numbering 45-528, not indices 0-483.
        c.label('pocket and name CA', '"%s%s" % (resn,resi)')
        c.orient('heme or ligand or pocket')
        c.zoom('heme or ligand or pocket',buffer=6)
        saved_view=c.get_view()
        # Session is useful for rotating and inspecting the authentic 200-state movie in desktop PyMOL.
        c.save(str(session))
        c.frame(1)
        render_path=frames_dir/'frame_0000.png'
        movie_frames=min(MAX_FRAMES,total)
        for k in range(movie_frames):
            frame=k+1
            c.frame(frame)
            c.set_view(saved_view)
            # Replace geometry measurements each frame, so yellow dashes follow moving atoms.
            for label,_,_ in INTERACTIONS[system]:
                c.delete('contact_'+label)
            for label,prot_atom,lig_atom in INTERACTIONS[system]:
                resi=int(''.join(ch for ch in label if ch.isdigit()))
                sel1=f'complex and resi {resi} and name {prot_atom} and polymer.protein'
                sel2=f'complex and resn LIG and name {lig_atom}'
                if c.count_atoms(sel1)==1 and c.count_atoms(sel2)==1:
                    dist=c.get_distance(sel1,sel2,state=frame)
                    if dist<=3.5:
                        c.distance('contact_'+label,sel1,sel2,cutoff=3.5,mode=0,state=frame)
                        c.hide('labels','contact_'+label)
            path=frames_dir/f'frame_{k:04d}.png'
            c.png(str(path),width=VIDEO_WIDTH,height=VIDEO_HEIGHT,ray=RAY_TRACE,quiet=1)
            time_ns=0.01+0.25*k
            add_caption(path,system,time_ns,k,movie_frames)
            if k==0:
                shutil.copyfile(path,preview)
                print(f'{system}: preview saved: {preview}')
            if (k+1)%25==0 or k+1==movie_frames:
                print(f'  {system}: rendered {k+1}/{movie_frames} genuine PyMOL cartoon frames',flush=True)
    finally:
        p.stop()
        display.stop()
    command=['ffmpeg','-y','-hide_banner','-loglevel','error',
             '-framerate',str(FPS),'-i',str(frames_dir/'frame_%04d.png'),
             '-vf',f'scale={VIDEO_WIDTH}:{VIDEO_HEIGHT},format=yuv420p',
             '-c:v','libx264','-preset','medium','-crf','20','-movflags','+faststart',
             str(video)]
    subprocess.run(command,check=True)
    if video.stat().st_size<10000:
        raise RuntimeError(f'{system}: MP4 seems invalid (too small)')
    print(f'{system}: FINAL CARTOON MP4: {video} ({video.stat().st_size/1024/1024:.1f} MiB)')
    return video,preview,session

# Render each compound separately. Results remain in Drive even if the other fails.
movies={}
for system in SYSTEMS:
    print('\n=== TRUE PROTEIN CARTOON ANIMATION:',system,'===')
    movies[system]=produce_cartoon(system,*source_files[system])
print('\nDONE — protein ribbons / ligand sticks / heme / dynamic polar contacts')


In [ ]:
# Cell 5 — Playback and paths.
from IPython.display import display, Video
for system in SYSTEMS:
    if system in movies:
        movie,preview,session=movies[system]
        print(system, 'MP4:', movie)
        print('Interactive PyMOL session:', session)
        display(Video(str(movie),embed=True,width=720))
